# Mining Footprint Mapping with Text Prompt Segmentation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/mining_footprint_mapping.ipynb)

This notebook demonstrates how to delineate open-pit mine footprints from aerial imagery using [CLIPSegmentation](https://huggingface.co/docs/transformers/model_doc/clipseg) — a zero-shot text-guided segmentation model. No labelled training data or domain-specific model is required; a plain-language text prompt such as `"open pit mine"` is sufficient to isolate the mine pit from surrounding terrain.

**Study area:** Bingham Canyon Copper Mine, Utah, USA — one of the largest open-pit mines in the world (~4 km wide, ~1.2 km deep).

**Outline:**
- Download NAIP aerial imagery via Planetary Computer
- Initialise a CLIPSegmentation model
- Generate a mine-pit probability mask using a text prompt
- Vectorise the mask and visualise the mine footprint

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [ ]:
# %pip install geoai-py

## Import libraries

In [ ]:
import geoai

## Create an interactive map

Centre the map on the Bingham Canyon Mine in Utah. You can pan and zoom to adjust the area of interest before downloading imagery.

In [ ]:
m = geoai.LeafMap(center=[40.5237, -112.1480], zoom=13)
m.add_basemap("Esri.WorldImagery")
m

## Define area of interest

The bounding box below covers the Bingham Canyon open-pit and its immediate surroundings. Replace it with `m.user_roi_bounds()` if you drew a custom rectangle on the map above.

In [ ]:
# [west, south, east, north] in WGS84 decimal degrees
if m.user_roi is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = (-112.2200, 40.5050, -112.0800, 40.5850)

## Download NAIP imagery

Download a National Agriculture Imagery Program (NAIP) scene from Planetary Computer. NAIP provides ~0.6 m/pixel 4-band (R, G, B, NIR) imagery across the continental United States.

In [ ]:
naip_paths = geoai.download_naip(
    bbox=bbox,
    output_dir="naip_bingham",
    year=2020,
    max_items=1,
)
raster_path = naip_paths[0]
print(f"Downloaded: {raster_path}")

## Inspect raster metadata

In [ ]:
geoai.print_raster_info(raster_path)

## Visualise imagery

Display the natural-colour composite (R-G-B bands 1-2-3) on an interactive map.

In [ ]:
geoai.view_raster(raster_path, layer_name="NAIP Imagery")

## Initialise CLIPSegmentation model

CLIPSeg is a vision-language model that segments objects described by free-form text. The model runs on CPU if no GPU is available, though GPU inference is substantially faster for large rasters.

In [ ]:
segmenter = geoai.CLIPSegmentation(tile_size=512, overlap=32)

## Segment mine footprint using text prompt

Pass a natural-language description of the target feature. CLIPSeg scores each image patch against the prompt and produces a probability mask — no labelled samples required.

In [ ]:
mask_path = "mine_mask.tif"
text_prompt = "open pit mine"

In [ ]:
segmenter.segment_image(
    raster_path,
    output_path=mask_path,
    text_prompt=text_prompt,
    threshold=0.4,
    smoothing_sigma=1.5,
)

## Visualise segmentation mask

Overlay the mine probability mask on the NAIP imagery. Brighter colours indicate higher confidence that a pixel belongs to the mine pit.

In [ ]:
geoai.view_raster(
    mask_path,
    nodata=0,
    opacity=0.7,
    colormap="hot",
    layer_name="Mine Mask",
    basemap=raster_path,
)

## Side-by-side comparison

Use a split map to compare the original imagery with the segmentation output.

In [ ]:
geoai.create_split_map(
    left_layer=mask_path,
    right_layer=raster_path,
    left_label="Mine Mask",
    right_label="NAIP Imagery",
    left_args={"nodata": 0, "opacity": 0.8, "colormap": "hot"},
    basemap=raster_path,
)

## Vectorise mine footprint

Convert the raster mask to vector polygons for use in GIS workflows. Small fragments below `min_area` are discarded to retain only the main pit boundary.

In [ ]:
gdf = geoai.orthogonalize(
    input_path=mask_path,
    output_path="mine_footprint.geojson",
    epsilon=0.5,
)
print(f"Polygons extracted: {len(gdf)}")

## Inspect results

In [ ]:
gdf.head()

In [ ]:
geoai.view_vector_interactive(
    gdf,
    style_kwds={"color": "#E8441A", "fillOpacity": 0.2, "weight": 2},
    layer_name="Mine Footprint",
    tiles=raster_path,
)

## Summary

This notebook showed how to:

1. Download NAIP aerial imagery for a mine area using `geoai.download_naip`
2. Apply zero-shot text-prompt segmentation with `geoai.CLIPSegmentation` — no labelled data required
3. Use the prompt `"open pit mine"` to isolate the pit from surrounding terrain
4. Vectorise the raster mask and visualise the extracted footprint

The same pipeline can be adapted to other extractive sites (quarries, tailing ponds, coal surface mines) by changing the `text_prompt` and `bbox`. For time-series mine expansion analysis, run the pipeline on multiple acquisition years and overlay the resulting footprint polygons.

## References

- [CLIPSeg: Image Segmentation Using Text and Image Prompts](https://arxiv.org/abs/2112.10003) — Lüddecke & Ecker, 2022
- [geoai](https://geoai.gishub.org/) — Wu et al.
- [NAIP on Planetary Computer](https://planetarycomputer.microsoft.com/dataset/naip)
- [Bingham Canyon Mine](https://en.wikipedia.org/wiki/Bingham_Canyon_Mine)